# Read-only Iceberg HadoopCatalog exploration

This notebook inspects a persistent HadoopCatalog created by the Java loader. It deliberately contains no write, maintenance, or catalog-mutation commands.

For local use, start the synthetic stack with `--runtime-dir .local-notebook` and set `ICEBERG_WAREHOUSE` as described in `notebooks/README.md`. GCS requires both `NOTEBOOK_ENABLE_GCS=true` and an explicit `gs://` warehouse.

In [ ]:
from pathlib import Path
import os
import sys

# Keep the notebook runnable directly from a source checkout.
repository_root = Path.cwd()
sys.path.insert(0, str(repository_root / 'src'))

from gce_hadoop_catalog.spark_catalog import create_spark_session, settings_from_environment

settings = settings_from_environment()
print({
    'catalog': settings.catalog_name,
    'warehouse': settings.warehouse,
    'namespace': settings.namespace,
    'table': settings.table,
    'gcs_enabled': settings.gcs_enabled,
})
spark = create_spark_session(settings)
spark.conf.get('spark.sql.session.timeZone')

## Discover the catalog

`NOTEBOOK_NAMESPACE` and `NOTEBOOK_TABLE` override the defaults (`alpaca` and `bars_raw`) without editing this notebook.

In [ ]:
spark.sql(f'SHOW NAMESPACES IN {settings.catalog_name}').show(truncate=False)
spark.sql(f'SHOW TABLES IN {settings.catalog_name}.{settings.namespace}').show(truncate=False)

In [ ]:
table = settings.table_identifier
spark.sql(f'DESCRIBE TABLE {table}').show(truncate=False)
spark.sql(f'SELECT committed_at, snapshot_id, operation, summary FROM {table}.snapshots ORDER BY committed_at DESC').show(truncate=False)

## Bounded bar query

The table stores UTC RFC 3339 timestamps in `t`, so the lexical range below is UTC-safe. Override `NOTEBOOK_START_UTC`, `NOTEBOOK_END_UTC`, and `NOTEBOOK_LIMIT` for another bounded inspection.

In [ ]:
start_utc = os.environ.get('NOTEBOOK_START_UTC', '2026-01-01T00:00:00Z')
end_utc = os.environ.get('NOTEBOOK_END_UTC', '2026-01-03T00:00:00Z')
limit = int(os.environ.get('NOTEBOOK_LIMIT', '100'))

bars = spark.sql(
    f"""
    SELECT symbol, t, o, h, l, c, v, n, vw, received_at, payload_hash
    FROM {table}
    WHERE t >= '{start_utc}' AND t < '{end_utc}'
    ORDER BY t, symbol
    LIMIT {limit}
    """
)
bars.show(truncate=False)

In [ ]:
spark.sql(
    f"EXPLAIN FORMATTED SELECT * FROM {table} WHERE t >= '{start_utc}' AND t < '{end_utc}'"
).show(truncate=False)